# 1 — EDA (sampled, memory-safe)

Runs AFTER `00_setup_colab.ipynb`. Uses small samples + chunk reads so we never OOM, even on 500MB files.

In [ ]:
%pip install -q pandas numpy matplotlib seaborn rapidfuzz tqdm
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/AMAZON_ML'
TRAIN = f'{BASE}/data/student_resource/dataset/train'
TEST  = f'{BASE}/data/student_resource/dataset/test'
WORK  = f'{BASE}/work'
print('ok')

In [ ]:
import pandas as pd
# Light samples for EDA
s1 = pd.read_csv(f'{TRAIN}/train_source1.tsv', sep='\t', nrows=20000)
s2 = pd.read_csv(f'{TRAIN}/train_source2.tsv', sep='\t', nrows=20000)
s3 = pd.read_csv(f'{TRAIN}/train_source3.tsv', sep='\t', nrows=20000)
gt = pd.read_csv(f'{TRAIN}/train_ground_truth.tsv', sep='\t', nrows=20000)
print(s1.shape, s2.shape, s3.shape, gt.shape)
print(s1.head(3).to_string())
print('countries train s1:', s1['country'].value_counts().to_dict())
t1 = pd.read_csv(f'{TEST}/test_source1.tsv', sep='\t', nrows=5000)
print('countries test s1 sample:', t1['country'].value_counts().to_dict())
print('nulls s1:', s1.isna().sum().to_dict())
# match-count distribution on sample
gt['n_matches'] = gt['matched_entity_ids'].fillna('').apply(lambda x: 0 if x=='' else len(str(x).split(',')))
print(gt['n_matches'].describe())
print(gt['n_matches'].value_counts().head(10))

In [ ]:
# Name/address noise examples: join a few ground-truth pairs and eyeball them
import pandas as pd
s1 = pd.read_csv(f'{TRAIN}/train_source1.tsv', sep='\t', nrows=50000).set_index('entity_id')
s2 = pd.read_csv(f'{TRAIN}/train_source2.tsv', sep='\t', nrows=100000).set_index('entity_id')
s3 = pd.read_csv(f'{TRAIN}/train_source3.tsv', sep='\t', nrows=100000).set_index('entity_id')
gt = pd.read_csv(f'{TRAIN}/train_ground_truth.tsv', sep='\t', nrows=200)
both = {**s2.to_dict('index'), **s3.to_dict('index')}
for _, row in gt.head(15).iterrows():
    print('='*100)
    print('S1:', row['source1_entity_id'], '|', s1.loc[row['source1_entity_id']].to_dict() if row['source1_entity_id'] in s1.index else 'S1 not in sample')
    for m in str(row['matched_entity_ids']).split(',')[:3]:
        m=m.strip()
        if m in both: print('  MTCH:', m, '|', both[m])
        else: print('  MTCH:', m, '| (not in sample)')
# SAVE finding: write your observations into Documentation_template.md section 2.1 later